# AMPS I · Week 04 — HW4: Interpolation convergence rates
**Due:** before the Week 5 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Proofs (25 pts)
$\mathscr I$ denotes P1 Lagrange interpolation.

**(a) (10)** For $u\in C^2([0,1])$ and $e=u-\mathscr Iu$ on the single cell $[0,1]$, prove
$\;\|e'\|_{L^2(0,1)}\le\|u''\|_{L^2(0,1)}$ and $\|e\|_{L^2(0,1)}\le\|u''\|_{L^2(0,1)}$.
*(Hint: Rolle's theorem gives a point where $e'=0$; then Cauchy–Schwarz.)*

**(b) (8)** By the change of variables $x=a+\ell t$, deduce on a cell $[a,b]$ of length $\ell$:
$\|e\|_{L^2(a,b)}\le\ell^2\|u''\|_{L^2(a,b)}$ and $|e|_{H^1(a,b)}\le\ell\,\|u''\|_{L^2(a,b)}$. Then state and prove the global estimates on a mesh with $h=\max\ell$.

**(c) (7)** In 2D let $K=F(\hat K)$ with $F(\hat x)=x_0+hR\hat x$ ($R$ a rotation), and $\hat v=v\circ F$. Show
$\|v\|_{L^2(K)}=h\|\hat v\|_{L^2(\hat K)}$, $|v|_{H^1(K)}=|\hat v|_{H^1(\hat K)}$, $|v|_{H^2(K)}=h^{-1}|\hat v|_{H^2(\hat K)}$.
Use these and the reference estimate $|\hat e|_{H^i(\hat K)}\le\hat C|\hat u|_{H^2(\hat K)}$ ($i=0,1$) to explain why the $H^1$ rate is one lower than the $L^2$ rate.

> *your proofs*

## Q2 — Convergence study (35 pts)
**(a) (12)** For $u_1=e^{x}\sin(3y)$ on the unit square: interpolate into P1, P2, P3 on $N\times N$ meshes, $N=4,8,16,32,64$.
Tabulate $\|u-\mathscr I_hu\|_{L^2}$, $|u-\mathscr I_hu|_{H^1}$ and the rates; plot them (log–log, $h$ on the $x$-axis labelled $1/N$).
Include the polynomial-preservation check for your code.

**(b) (12)** Repeat for $u_2=x^{3/4}$ (singular along the whole edge $x=0$, not just at a corner).
*Before* computing, predict the rates: a 1D function $x^\alpha$ belongs to $H^s(0,1)$ exactly for $s<\alpha+\tfrac12$, and $u_2$ does not depend on $y$.
Compare the prediction with your numbers for all three degrees. *(Careful: some DOF coordinates on $x=0$ are slightly negative.)*

**(c) (11)** Grade the mesh towards the singular edge: after creating the mesh, set
`msh.geometry.x[:, 0] = msh.geometry.x[:, 0] ** gamma`. For P1 and P2 and $\gamma\in\{1,3,4,6\}$, report the $H^1$ rates (in terms of $N$).
Which $\gamma$ restores the optimal rate for P1? For P2? Explain in one or two sentences why grading helps.

In [ ]:
%%fenicsx -np 1
# TODO: (a) u1 for k = 1, 2, 3, N = 4..64; polynomial-preservation check; save errors to /tmp/fnx/hw4_q2.npz
# TODO: (b) u2 = x^(3/4)  (c) graded meshes, gamma in {1, 3, 4, 6}, k = 1, 2

In [ ]:
# TODO: load /tmp/fnx/hw4_q2.npz and plot the errors of (a) and (b) on log–log axes

> *your answers to (b) and (c)*

## Q3 — Which triangles are harmful? (20 pts)
The scaling proof (lecture §A7) needs $h_K/\rho_K$ bounded for $H^1$ estimates. Test whether that is really necessary.
Use **one** triangle (`mesh.create_mesh` with a single cell) and P1:

- *needle*: vertices $(0,0),(1,0),(0,\varepsilon)$ — one angle $\to0$, the largest angle stays $90^\circ$;
- *flat*: vertices $(-1,0),(1,0),(0,\varepsilon)$ — the largest angle $\to180^\circ$.

Both have $h_K/\rho_K\to\infty$ as $\varepsilon\to0$.

**(a) (12)** For $\varepsilon\in\{1,10^{-1},10^{-2},10^{-3}\}$ and $u\in\{x^2,\ y^2,\ xy\}$ compute $|u-\mathscr Iu|_{H^1(K)}/|u|_{H^2(K)}$. Tabulate.

**(b) (8)** Is a bounded $h_K/\rho_K$ necessary? Which geometric property actually matters? Explain the flat case by hand for $u=x^2$
(what is $\mathscr Iu$?). What does this mean for meshes near walls or thin layers?

In [ ]:
%%fenicsx -np 1
# TODO: one-triangle meshes, P1 interpolation of x^2, y^2, xy; ratio |u - I u|_H1 / |u|_H2

> *your answers to (b)*

## Q4 — LLM-assisted: transfer between non-matching meshes (20 pts)
Ask an LLM (with the API sheet) for DOLFINx 0.11 code that: builds a fine triangle mesh (32×32) and a coarse **quadrilateral** mesh (5×7),
interpolates a function into P2 on the fine mesh, **transfers** it to P2 on the coarse mesh, and writes the coarse field to a file
ParaView can open.

**(a) (8)** Paste the unchanged generated code in the Markdown cell and your corrected, working version in the code cell. In the AI log,
list what was wrong.

**(b) (12)** Verify: (i) a quadratic must arrive exactly (error $\lesssim10^{-14}$) — why? (ii) for $u=\sin(\pi x)\sin(\pi y)$ compare the
transferred field with direct interpolation of $u$ on the coarse mesh (how large is the difference, and why is it not zero?);
(iii) the result is the same with `-np 1` and `-np 2`. If `dolfinx.has_adios2` is false on your runtime, write XDMF instead (which degree?).

**(a) generated code (unchanged)**

```python
# paste here
```

In [ ]:
%%fenicsx -np 2
# TODO: corrected code; checks (i)–(iii)

> *(b) your answers*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …